# Разбиение Java-Pair датасета по commit hash

Один `commit_hash` целиком попадает только в train или только в test. Это предотвращает leakage между связанными файлами одного commit.

Ноутбук создаёт split отдельно для FILE-level и PROJECT-level форматов.

In [ ]:
from pathlib import Path
import json
import random

## 1. Настройки

Измени `TEST_FRACTION` или `RANDOM_SEED`, если нужен другой split.

In [ ]:
DATASET_DIR = Path('/Users/nident/Desktop/JOB/ScolTech/CLEAR_Causal_Context_based_Agentic_Reasoning_for_Vulnerability_Detection2/dataset/java-pair')
FILE_INPUT = DATASET_DIR / 'java_data_paired_examples_FILE_LVL.json'
PROJECT_INPUT = DATASET_DIR / 'java_data_paired_examples_PROJECT_LVL.json'
TEST_FRACTION = 0.20
RANDOM_SEED = 42

print('Dataset directory:', DATASET_DIR)
print('Test fraction:', TEST_FRACTION)
print('Random seed:', RANDOM_SEED)

## 2. Загрузка данных

In [ ]:
def load_json(path):
    with path.open(encoding='utf-8') as file:
        return json.load(file)

file_rows = load_json(FILE_INPUT)
project_rows = load_json(PROJECT_INPUT)
print('FILE-level rows:', len(file_rows))
print('PROJECT-level rows:', len(project_rows))

## 3. Удаление записей без commit hash

Запись без commit hash нельзя корректно распределить по commit-группам. Такие записи сохраняются в отчёте, но не попадают в train/test.

In [ ]:
def has_commit(row):
    return bool(str(row.get('commit_hash', '')).strip())

file_valid = [row for row in file_rows if has_commit(row)]
project_valid = [row for row in project_rows if has_commit(row)]
file_excluded = [row for row in file_rows if not has_commit(row)]
project_excluded = [row for row in project_rows if not has_commit(row)]

print('Excluded FILE-level rows:', len(file_excluded))
print('Excluded PROJECT-level rows:', len(project_excluded))

## 4. Создание общего списка commit hash

FILE-level и PROJECT-level используют один и тот же commit split. Поэтому связанные записи гарантированно находятся в одной части.

In [ ]:
file_commits = {row['commit_hash'] for row in file_valid}
project_commits = {row['commit_hash'] for row in project_valid}
if file_commits != project_commits:
    raise ValueError('FILE-level и PROJECT-level содержат разные commit hash')

commits = sorted(file_commits)
random_generator = random.Random(RANDOM_SEED)
random_generator.shuffle(commits)
test_count = round(len(commits) * TEST_FRACTION)
test_commits = set(commits[:test_count])
train_commits = set(commits[test_count:])

assert not train_commits & test_commits
print('Total commits:', len(commits))
print('Train commits:', len(train_commits))
print('Test commits:', len(test_commits))

## 5. Разделение строк по группам commit

In [ ]:
def split_rows(rows):
    train = [row for row in rows if row['commit_hash'] in train_commits]
    test = [row for row in rows if row['commit_hash'] in test_commits]
    return train, test

file_train, file_test = split_rows(file_valid)
project_train, project_test = split_rows(project_valid)

print('FILE train/test:', len(file_train), len(file_test))
print('PROJECT train/test:', len(project_train), len(project_test))

## 6. Сохранение JSON и списков commit

In [ ]:
def save_json(name, data):
    path = DATASET_DIR / name
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return path

def save_commits(name, commits_to_save):
    path = DATASET_DIR / name
    path.write_text('\n'.join(sorted(commits_to_save)) + '\n', encoding='utf-8')
    return path

outputs = {
    'java_train_paired_FILE_LVL.json': file_train,
    'java_test_paired_FILE_LVL.json': file_test,
    'java_train_paired_PROJECT_LVL.json': project_train,
    'java_test_paired_PROJECT_LVL.json': project_test,
}
for name, data in outputs.items():
    print('Saved:', save_json(name, data))

save_commits('train_commits.txt', train_commits)
save_commits('test_commits.txt', test_commits)
save_json('split_report.json', {
    'random_seed': RANDOM_SEED,
    'test_fraction': TEST_FRACTION,
    'total_commits': len(commits),
    'train_commits': len(train_commits),
    'test_commits': len(test_commits),
    'file_train_rows': len(file_train),
    'file_test_rows': len(file_test),
    'project_train_rows': len(project_train),
    'project_test_rows': len(project_test),
    'excluded_file_rows': len(file_excluded),
    'excluded_project_rows': len(project_excluded),
})

## 7. Проверка отсутствия leakage

In [ ]:
file_train_ids = {row['commit_hash'] for row in file_train}
file_test_ids = {row['commit_hash'] for row in file_test}
project_train_ids = {row['commit_hash'] for row in project_train}
project_test_ids = {row['commit_hash'] for row in project_test}

assert not file_train_ids & file_test_ids
assert not project_train_ids & project_test_ids
assert file_train_ids == train_commits
assert file_test_ids == test_commits
assert project_train_ids == train_commits
assert project_test_ids == test_commits
print('Проверка пройдена: commit leakage отсутствует.')